**STEP 1**

In [2]:
import os
import requests
import time

output_folder = "html_pages"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

base_url = "https://nguoikesu.com/nhan-vat"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

total_pages = 291
items_per_page = 5 # Dựa trên cấu trúc ?start=5, 10, 15...

for i in range(0, total_pages):
    # Tính toán tham số start: Trang 1 (start=0), Trang 2 (start=5)...
    start_param = i * items_per_page
    url = f"{base_url}?start={start_param}"
    
    print(f"Đang tải Trang {i+1}/{total_pages} (start={start_param})...")
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            file_path = os.path.join(output_folder, f"page_{i+1}.txt")
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(response.text)
        else:
            print(f"Lỗi HTTP {response.status_code} tại trang {i+1}")
        
        # Nghỉ một chút để tránh bị chặn (Rate limit)
        time.sleep(1) 
    except Exception as e:
        print(f"Lỗi kết nối tại trang {i+1}: {e}")

print("Xong giai đoạn 1: Đã lưu các file vào folder html_pages")

Đang tải Trang 1/291 (start=0)...
Đang tải Trang 2/291 (start=5)...
Đang tải Trang 3/291 (start=10)...
Đang tải Trang 4/291 (start=15)...
Đang tải Trang 5/291 (start=20)...
Đang tải Trang 6/291 (start=25)...
Đang tải Trang 7/291 (start=30)...
Đang tải Trang 8/291 (start=35)...
Đang tải Trang 9/291 (start=40)...
Đang tải Trang 10/291 (start=45)...
Đang tải Trang 11/291 (start=50)...
Đang tải Trang 12/291 (start=55)...
Đang tải Trang 13/291 (start=60)...
Đang tải Trang 14/291 (start=65)...
Đang tải Trang 15/291 (start=70)...
Đang tải Trang 16/291 (start=75)...
Đang tải Trang 17/291 (start=80)...
Đang tải Trang 18/291 (start=85)...
Đang tải Trang 19/291 (start=90)...
Đang tải Trang 20/291 (start=95)...
Đang tải Trang 21/291 (start=100)...
Đang tải Trang 22/291 (start=105)...
Đang tải Trang 23/291 (start=110)...
Đang tải Trang 24/291 (start=115)...
Đang tải Trang 25/291 (start=120)...
Đang tải Trang 26/291 (start=125)...
Đang tải Trang 27/291 (start=130)...
Đang tải Trang 28/291 (start=135

**STEP 2**

In [3]:
import os
import json
from bs4 import BeautifulSoup

def process_folder_to_json(folder_path):
    final_data = []
    
    # Lấy danh sách file và sắp xếp theo số trang 1, 2, 3...
    files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
    files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        print(f"Đang xử lý cấu trúc: {filename}")
        
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            soup = BeautifulSoup(content, 'html.parser')
            
            # 1. Lấy toàn bộ nội dung text của trang đó (all_content)
            page_all_text = soup.get_text(separator=' ', strip=True)
            
            # 2. Trích xuất danh sách nhân vật trong trang
            # Dựa trên cấu trúc NguoiKeSu: Mỗi nhân vật nằm trong div class "item-page" hoặc "page-header"
            # Ở trang danh sách, các nhân vật thường nằm trong h2
            items = soup.find_all(['h2', 'div'], class_=['page-header', 'item-page'])
            
            characters_in_page = []
            for item in items:
                a_tag = item.find('a')
                if a_tag and a_tag.get('href'):
                    name = a_tag.get_text(strip=True)
                    if not name: continue
                    
                    # Tìm mô tả ngay sau tiêu đề (thường ở thẻ p hoặc div tiếp theo)
                    summary = ""
                    next_node = item.find_next_sibling()
                    if next_node:
                        summary = next_node.get_text(strip=True)

                    characters_in_page.append({
                        "name": name,
                        "link": "https://nguoikesu.com" + a_tag['href'],
                        "short_summary": summary
                    })

            # Lưu vào cấu trúc tổng
            final_data.append({
                "page_number": filename.replace(".txt", ""),
                "all_content": page_all_text,
                "characters": characters_in_page
            })

    # Xuất file JSON tổng hợp
    with open("nguoikesu_full_database.json", "w", encoding="utf-8") as f:
        json.dump(final_data, f, ensure_ascii=False, indent=4)

if __name__ == "__main__":
    process_folder_to_json("html_pages")
    print("\n--- HOÀN THÀNH ---")
    print("Dữ liệu đã được cấu trúc hóa vào file: nguoikesu_full_database.json")

Đang xử lý cấu trúc: page_1.txt
Đang xử lý cấu trúc: page_2.txt
Đang xử lý cấu trúc: page_3.txt
Đang xử lý cấu trúc: page_4.txt
Đang xử lý cấu trúc: page_5.txt
Đang xử lý cấu trúc: page_6.txt
Đang xử lý cấu trúc: page_7.txt
Đang xử lý cấu trúc: page_8.txt
Đang xử lý cấu trúc: page_9.txt
Đang xử lý cấu trúc: page_10.txt
Đang xử lý cấu trúc: page_11.txt
Đang xử lý cấu trúc: page_12.txt
Đang xử lý cấu trúc: page_13.txt
Đang xử lý cấu trúc: page_14.txt
Đang xử lý cấu trúc: page_15.txt
Đang xử lý cấu trúc: page_16.txt
Đang xử lý cấu trúc: page_17.txt
Đang xử lý cấu trúc: page_18.txt
Đang xử lý cấu trúc: page_19.txt
Đang xử lý cấu trúc: page_20.txt
Đang xử lý cấu trúc: page_21.txt
Đang xử lý cấu trúc: page_22.txt
Đang xử lý cấu trúc: page_23.txt
Đang xử lý cấu trúc: page_24.txt
Đang xử lý cấu trúc: page_25.txt
Đang xử lý cấu trúc: page_26.txt
Đang xử lý cấu trúc: page_27.txt
Đang xử lý cấu trúc: page_28.txt
Đang xử lý cấu trúc: page_29.txt
Đang xử lý cấu trúc: page_30.txt
Đang xử lý cấu trúc

**STEP 3**

In [4]:
import os
import json
import requests
import time
import re

def download_character_details(json_file):
    # 1. Đọc dữ liệu từ file JSON bạn đã tạo ở bước trước
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Lỗi khi đọc file JSON: {e}")
        return

    # 2. Tạo folder lưu trữ chi tiết nhân vật
    output_folder = "characters_details"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    count = 0
    # Duyệt qua từng trang trong database
    for page in data:
        print(f"--- Đang xử lý các nhân vật tại {page['page_number']} ---")
        
        for char in page['characters']:
            name = char['name']
            link = char['link']
            
            # Làm sạch tên để đặt tên file (xóa ký tự đặc biệt)
            safe_name = re.sub(r'[\\/*?:"<>|]', "", name)
            file_path = os.path.join(output_folder, f"{safe_name}.txt")
            
            # Kiểm tra nếu file đã tồn tại thì bỏ qua (tiết kiệm thời gian nếu chạy lại)
            if os.path.exists(file_path):
                continue

            try:
                print(f"Đang tải chi tiết: {name}...")
                response = requests.get(link, headers=headers, timeout=15)
                
                if response.status_code == 200:
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(response.text)
                    count += 1
                else:
                    print(f"Lỗi HTTP {response.status_code} với nhân vật: {name}")

                # Nghỉ một chút để tránh bị server chặn
                time.sleep(0.5) 
                
            except Exception as e:
                print(f"Lỗi khi truy cập {name}: {e}")

    print(f"\n--- HOÀN THÀNH ---")
    print(f"Đã tải xong {count} file nhân vật vào folder '{output_folder}'")

if __name__ == "__main__":
    # Đảm bảo file nguoikesu_full_database.json nằm cùng thư mục
    download_character_details("nguoikesu_full_database.json")

--- Đang xử lý các nhân vật tại page_1 ---
Đang tải chi tiết: An Dương Vương...
Đang tải chi tiết: Âu Cơ...
Đang tải chi tiết: Bà Triệu...
Đang tải chi tiết: Bảo Đại...
Đang tải chi tiết: Bộ Chất...
--- Đang xử lý các nhân vật tại page_2 ---
Đang tải chi tiết: Bùi Bá Kỳ...
Đang tải chi tiết: Bùi Bị...
Đang tải chi tiết: Bùi Cầm Hổ...
Đang tải chi tiết: Bùi Diễm...
Đang tải chi tiết: Bùi Dương Lịch...
--- Đang xử lý các nhân vật tại page_3 ---
Đang tải chi tiết: Bùi Đắc Tuyên...
Đang tải chi tiết: Bùi Huy Bích...
Đang tải chi tiết: Bùi Kỷ...
Đang tải chi tiết: Bùi Minh Quốc...
Đang tải chi tiết: Bùi Mộc Đạc...
--- Đang xử lý các nhân vật tại page_4 ---
Đang tải chi tiết: Bùi Nam Hà...
Đang tải chi tiết: Bùi Quang Tạo...
Đang tải chi tiết: Bùi Quang Thận...
Đang tải chi tiết: Bùi Quốc Hưng...
Đang tải chi tiết: Bùi Quốc Khái...
--- Đang xử lý các nhân vật tại page_5 ---
Đang tải chi tiết: Bùi Tá Hán...
Đang tải chi tiết: Bùi Thế Đạt...
Đang tải chi tiết: Bùi Thị Nhạn...
Đang tải chi tiết

In [13]:
import os
import json
from bs4 import BeautifulSoup

def parse_character_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        html_content = f.read()

    soup = BeautifulSoup(html_content, 'html.parser')

    # 1. Lấy tên nhân vật chuẩn
    # Ưu tiên h2 trong itemprop="name" (đặc trưng của trang chi tiết NguoiKeSu)
    name = "Không rõ"
    header_element = soup.find('h2', itemprop='name') or \
                     (soup.find('div', class_='page-header').find('h2') if soup.find('div', class_='page-header') else None)
    
    if header_element:
        # Sử dụng separator để tránh dính chữ ngay từ tiêu đề
        name_raw = header_element.get_text(separator=' ', strip=True)
        # Loại bỏ cụm từ "Nhân Vật Lịch Sử" nếu nó bị dính vào tên
        name = name_raw.replace("Nhân Vật Lịch Sử", "").strip()
        name = " ".join(name.split())
    else:
        # Dự phòng: Lấy từ thẻ title
        title_tag = soup.find('title')
        if title_tag:
            name = title_tag.get_text().split('-')[0].strip()

    # 2. Lấy Infobox (Dữ liệu bảng tóm tắt bên phải)
    infobox_data = {}
    infobox = soup.find('table', class_='infobox')
    if infobox:
        for row in infobox.find_all('tr'):
            label = row.find(['th', 'td'], class_='infobox-label') or row.find('th')
            value = row.find('td', class_='infobox-data') or row.find('td')
            
            if label and value:
                key = " ".join(label.get_text(separator=' ', strip=True).split())
                val = " ".join(value.get_text(separator=' ', strip=True).split())
                # Tránh lấy nhầm các hàng tiêu đề bảng (key trùng value)
                if key and val and key != val:
                    infobox_data[key] = val

    # 3. Lấy Biography (Danh sách các chuỗi văn bản thuần túy)
    article_container = soup.find('div', itemprop='articleBody') or soup.find('div', class_='item-page')
    biography_list = []
    
    if article_container:
        # Quét p, h2, h3, h4, li (bao gồm cả các đoạn dẫn và chú thích)
        elements = article_container.find_all(['p', 'h2', 'h3', 'h4', 'li'], recursive=True)
        
        for el in elements:
            # Bỏ qua nếu thuộc bảng Infobox để tránh lặp lại dữ liệu
            if el.find_parent('table', class_='infobox'):
                continue
            
            # Bỏ qua các thành phần rác điều hướng
            if any(cls in el.get('class', []) for cls in ['tags', 'article-info', 'aside']):
                continue

            # Xử lý lỗi dính chữ bằng separator=' ' và dọn dẹp khoảng trắng thừa
            text = " ".join(el.get_text(separator=' ', strip=True).split())
            
            # Lọc bỏ tên nhân vật lặp lại trong bài hoặc chữ "Nhân Vật Lịch Sử"
            if text and text not in ["Nhân Vật Lịch Sử", name] and text not in biography_list:
                biography_list.append(text)

    return {
        "name": name,
        "infobox": infobox_data,
        "biography": biography_list,
        "all_content": "\n\n".join(biography_list)
    }

def process_full_folder(input_folder, output_json):
    all_data = []
    if not os.path.exists(input_folder):
        print(f"Lỗi: Thư mục '{input_folder}' không tồn tại.")
        return

    # Lấy danh sách file .txt
    files = [f for f in os.listdir(input_folder) if f.endswith('.txt')]
    total = len(files)
    print(f"Bắt đầu xử lý {total} file nhân vật...")

    for index, filename in enumerate(files, 1):
        path = os.path.join(input_folder, filename)
        try:
            char_info = parse_character_file(path)
            # Thêm metadata về file gốc để đối chiếu nếu cần
            char_info["source_file"] = filename
            all_data.append(char_info)
            
            if index % 50 == 0 or index == total:
                print(f"Tiến độ: {index}/{total} (Đã xong: {char_info['name']})")
                
        except Exception as e:
            print(f"!!! Lỗi tại file {filename}: {e}")

    # Ghi toàn bộ dữ liệu vào 1 file JSON duy nhất
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(all_data, f, ensure_ascii=False, indent=4)
    
    print(f"\nHOÀN THÀNH!")
    print(f"Tổng số nhân vật đã lưu: {len(all_data)}")
    print(f"Kết quả lưu tại: {output_json}")

if __name__ == "__main__":
    # Thay đổi đường dẫn này đến folder chứa các file .txt của bạn
    FOLDER_CHUA_TXT = "characters_details" 
    FILE_JSON_KET_QUA = "nguoikesu_final_database.json"
    
    process_full_folder(FOLDER_CHUA_TXT, FILE_JSON_KET_QUA)

Bắt đầu xử lý 1451 file nhân vật...
Tiến độ: 50/1451 (Đã xong: Dương Hạo)
Tiến độ: 100/1451 (Đã xong: Hoàng Liên Sơn)
Tiến độ: 150/1451 (Đã xong: Hàm Nghi)
Tiến độ: 200/1451 (Đã xong: Lê Cung Hoàng)
Tiến độ: 250/1451 (Đã xong: Lê Lễ)
Tiến độ: 300/1451 (Đã xong: Lê Thận)
Tiến độ: 350/1451 (Đã xong: Lê Đạt)
Tiến độ: 400/1451 (Đã xong: Lý Nhân)
Tiến độ: 450/1451 (Đã xong: Lý Đạo Thành)
Tiến độ: 500/1451 (Đã xong: Nguyễn Chiến)
Tiến độ: 550/1451 (Đã xong: Nguyễn Hảo)
Tiến độ: 600/1451 (Đã xong: Nguyễn Mậu)
Tiến độ: 650/1451 (Đã xong: Nguyễn Sinh Hùng)
Tiến độ: 700/1451 (Đã xong: Nguyễn Thị Vân)
Tiến độ: 750/1451 (Đã xong: Nguyễn Văn Minh)
Tiến độ: 800/1451 (Đã xong: Nguyễn Đăng Đạo)
Tiến độ: 850/1451 (Đã xong: Ngô Xương Xí)
Tiến độ: 900/1451 (Đã xong: nhà Tần)
Tiến độ: 950/1451 (Đã xong: Phạm Công Trứ)
Tiến độ: 1000/1451 (Đã xong: Phạm Văn Trà)
Tiến độ: 1050/1451 (Đã xong: Trương Chi Động)
Tiến độ: 1100/1451 (Đã xong: Trần Duy Hưng)
Tiến độ: 1150/1451 (Đã xong: Trần Phong)
Tiến độ: 1200/14